# Part D: Food-order stream with RabbitMQ
**Assignment: From Git to Gold (20 marks)**

This notebook sets up a local RabbitMQ broker, runs the drivers in the background with unbuffered logging, runs the dispatcher for 2 minutes, and displays the consumed messages proving **D1 (Topic Routing)** and **D3 (ACK / NACK)**.


## 1. Install & Start RabbitMQ in Colab

In [ ]:
!apt-get install -y -qq rabbitmq-server > /dev/null
!service rabbitmq-server start
!pip install -q pika


## 2. Verify RabbitMQ Service Status

In [ ]:
!service rabbitmq-server status

## 3. Create the `stream/` Project Files

In [ ]:
!mkdir -p stream

In [ ]:
%%writefile stream/dispatcher.py
"""
stream/dispatcher.py
Zomato / Rapido-style food & ride dispatch simulator using RabbitMQ topic exchange.
Publishes a dispatch request every 2 seconds to the 'rapido-dispatch' topic exchange.
"""

import json
import os
import random
import time
import uuid
from datetime import datetime, timezone

import pika

EXCHANGE_NAME = "rapido-dispatch"
EXCHANGE_TYPE = "topic"

ZONES_BY_CITY = {
    "Bangalore": ["koramangala", "indiranagar", "whitefield"],
    "Chennai": ["tnagar", "whitefield"],
}
TYPES = ["food", "ride"]


def get_connection():
    url = os.getenv("RABBITMQ_URL")
    if url:
        return pika.BlockingConnection(pika.URLParameters(url))
    return pika.BlockingConnection(pika.ConnectionParameters(host="localhost"))


def run_dispatcher():
    print(f"[*] Starting dispatcher... publishing to exchange '{EXCHANGE_NAME}'", flush=True)
    connection = get_connection()
    channel = connection.channel()

    channel.exchange_declare(
        exchange=EXCHANGE_NAME,
        exchange_type=EXCHANGE_TYPE,
        durable=True,
    )

    message_count = 0

    try:
        while True:
            message_count += 1
            city = random.choice(list(ZONES_BY_CITY.keys()))
            zone = random.choice(ZONES_BY_CITY[city])
            order_type = random.choice(TYPES)

            # D3: Every 10th message is intentionally malformed (missing 'zone')
            is_bad_message = (message_count % 10 == 0)

            # Routing key: city.zone.type (lowercase)
            routing_key = f"{city.lower()}.{zone.lower()}.{order_type.lower()}"

            payload = {
                "id": str(uuid.uuid4())[:8],
                "city": city,
                "type": order_type,
                "ts": datetime.now(timezone.utc).isoformat(),
            }

            if is_bad_message:
                print(f"\n[!] Intentionally BAD message #{message_count:03d} (missing 'zone')", flush=True)
            else:
                payload["zone"] = zone

            message_body = json.dumps(payload)

            channel.basic_publish(
                exchange=EXCHANGE_NAME,
                routing_key=routing_key,
                body=message_body,
                properties=pika.BasicProperties(
                    delivery_mode=pika.DeliveryMode.Transient,
                    content_type="application/json",
                ),
            )

            status_tag = "BAD (NO ZONE)" if is_bad_message else "OK"
            print(f"[#{message_count:03d} {status_tag}] Published with key '{routing_key}': {message_body}", flush=True)
            time.sleep(2)

    except KeyboardInterrupt:
        print("\n[*] Dispatcher stopped.", flush=True)
    finally:
        connection.close()


if __name__ == "__main__":
    run_dispatcher()


In [ ]:
%%writefile stream/driver.py
"""
stream/driver.py
Driver consumer script with manual ACK / NACK handling.
"""

import json
import os
import sys
import pika

EXCHANGE_NAME = "rapido-dispatch"
EXCHANGE_TYPE = "topic"


def get_connection():
    url = os.getenv("RABBITMQ_URL")
    if url:
        return pika.BlockingConnection(pika.URLParameters(url))
    return pika.BlockingConnection(pika.ConnectionParameters(host="localhost"))


def main():
    if len(sys.argv) < 3:
        print("Usage: python driver.py <NAME> <PATTERN>", flush=True)
        sys.exit(1)

    driver_name = sys.argv[1]
    binding_pattern = sys.argv[2]

    try:
        connection = get_connection()
        channel = connection.channel()

        channel.exchange_declare(
            exchange=EXCHANGE_NAME,
            exchange_type=EXCHANGE_TYPE,
            durable=True,
        )

        queue_name = f"queue_{driver_name.lower()}"
        channel.queue_declare(queue=queue_name, exclusive=False, auto_delete=True)

        channel.queue_bind(
            exchange=EXCHANGE_NAME,
            queue=queue_name,
            routing_key=binding_pattern,
        )

        print(f"[{driver_name}] Listening for orders matching '{binding_pattern}' on queue '{queue_name}'...", flush=True)

        def callback(ch, method, properties, body):
            try:
                data = json.loads(body.decode("utf-8"))
            except Exception as e:
                print(f"[{driver_name}] [MALFORMED JSON] Error decoding: {e}", flush=True)
                ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)
                return

            order_id = data.get("id")
            city = data.get("city")
            zone = data.get("zone")
            order_type = data.get("type")

            # D3: Reject message missing 'zone' with requeue=False
            if not zone:
                print(
                    f"[{driver_name}] [REJECTED - NACK] Order #{order_id} missing 'zone'! "
                    f"Key: '{method.routing_key}'. Dropping (requeue=False).",
                    flush=True
                )
                ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)
            else:
                print(
                    f"[{driver_name}] [ACCEPTED - ACK] Order #{order_id} | "
                    f"{city} / {zone} / {order_type} | Key: '{method.routing_key}'",
                    flush=True
                )
                ch.basic_ack(delivery_tag=method.delivery_tag)

        channel.basic_consume(
            queue=queue_name,
            on_message_callback=callback,
            auto_ack=False,
        )

        channel.start_consuming()

    except KeyboardInterrupt:
        print(f"[{driver_name}] Stopped.", flush=True)
    except Exception as e:
        print(f"[{driver_name}] ERROR: {e}", flush=True)
        sys.exit(1)


if __name__ == "__main__":
    main()


## 4. Launch Drivers in Background (Part D2)
We start 4 drivers with `-u` (unbuffered stdout) so every print reaches the log files in real time.


In [ ]:
import subprocess, sys, time, os

drivers = [
    ("Ravi", "bangalore.koramangala.*"),
    ("Arjun", "bangalore.*.ride"),
    ("Meena", "bangalore.*.food"),
    ("Control", "bangalore.#")
]

processes = []
for name, pattern in drivers:
    log_path = f"{name.lower()}.log"
    # Ensure fresh log file
    if os.path.exists(log_path):
        os.remove(log_path)
    log_file = open(log_path, "w", buffering=1)  # line-buffered
    # -u ensures unbuffered standard output and standard error
    cmd = [sys.executable, "-u", "stream/driver.py", name, pattern]
    p = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
    processes.append((p, log_file, name))
    print(f"Started driver {name:7s} | pattern: '{pattern}' (PID {p.pid})")

time.sleep(2)

# Verify all drivers are alive and connected
for p, log_file, name in processes:
    if p.poll() is not None:
        print(f"ERROR: Driver {name} exited prematurely with exit code {p.poll()}!")
        with open(f"{name.lower()}.log") as f:
            print("Log content:", f.read())
    else:
        print(f"Driver {name:7s} is active and listening.")


## 5. Run Dispatcher for 2 Minutes (Part D2 & D3)
Publishes requests every 2s for 120 seconds (~60 messages). 
Every 10th message intentionally omits `zone` to verify D3 NACK behavior.


In [ ]:
import subprocess, sys, time

print("Starting dispatcher for 120 seconds (2 minutes)...")
# Run dispatcher with unbuffered output (-u)
p_disp = subprocess.Popen([sys.executable, "-u", "stream/dispatcher.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

start_time = time.time()
while time.time() - start_time < 120:
    line = p_disp.stdout.readline()
    if line:
        print(line, end="")
    time.sleep(0.05)

p_disp.terminate()
print("\n[*] 2-minute dispatcher run completed!")


## 6. Inspect Driver Logs: Verify Routing (D1/D2) and ACK/NACK (D3)
Proves:
1. `bangalore.koramangala.food` was received by **Ravi**, **Meena**, and **Control**.
2. Bad messages (missing `zone`) are tagged `[REJECTED - NACK]` and dropped.
3. Good messages are tagged `[ACCEPTED - ACK]`.


In [ ]:
import time

# Give 1 second for queues to settle
time.sleep(1)

# Stop driver processes cleanly
for p, log_file, name in processes:
    p.terminate()
    try:
        p.wait(timeout=2)
    except:
        pass
    log_file.flush()
    log_file.close()

for name, pattern in drivers:
    print(f"\n{'='*25} LOGS FOR {name.upper()} ({pattern}) {'='*25}")
    with open(f"{name.lower()}.log") as f:
        content = f.read()
        lines = [l for l in content.strip().split("\n") if l.strip()]
        print(f"Total entries logged: {len(lines)}")
        for l in lines[:20]:
            print("  ", l)
